# E2 — Frontiere di decisione

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emanbuc/corso_foundation_tabular/blob/main/notebooks/usare-tfm/E2_frontiere_decisione.ipynb)

**Corso 1 · Usare un TFM — Modulo 2 (laboratorio, 15 min)**

Su due dataset giocattolo in 2D si guarda la frontiera di decisione del TFM e la si confronta con quella di tre modelli classici.

**Idea da far emergere:** come cambia la frontiera passando da 20 a 500 punti di contesto.

- **Dati:** `make_moons` e `make_circles` di scikit-learn.
- **Modelli:** TFM (TabICLv2), regressione logistica, kNN, random forest.

Il notebook gira anche su CPU, più lentamente: in quel caso conviene ridurre `RISOLUZIONE`.

## 0. Setup

In [ ]:
%pip install -q tabicl==2.2.0

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import make_circles, make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from tabicl import TabICLClassifier

print("GPU disponibile:", torch.cuda.is_available())

## 1. I dati

Si generano 500 punti per dataset. I contesti più piccoli sono i primi `n` punti dello stesso insieme: così ogni riga della griglia aggiunge dati alla precedente.

In [ ]:
DIMENSIONI = [20, 50, 200, 500]   # punti di contesto
RISOLUZIONE = 80                  # punti per lato della griglia su cui si disegna la frontiera

X_moons, y_moons = make_moons(n_samples=500, noise=0.25, random_state=0)
X_circles, y_circles = make_circles(n_samples=500, noise=0.15, factor=0.5, random_state=0)

fig, assi = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, (nome, X, y) in zip(assi, [("moons", X_moons, y_moons), ("circles", X_circles, y_circles)]):
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu", edgecolors="k", s=12)
    ax.set_title(nome)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## 2. I modelli

Tutti con i parametri di default. Il TFM non ne ha da scegliere.

In [ ]:
def modelli():
    return {
        "TFM (TabICLv2)": TabICLClassifier(),
        "Regressione logistica": LogisticRegression(),
        "kNN (k = 5)": KNeighborsClassifier(n_neighbors=5),
        "Random forest": RandomForestClassifier(random_state=0),
    }

## 3. La griglia delle frontiere

Ogni riga è una dimensione del contesto, ogni colonna un modello. Il colore è la probabilità predetta della classe 1, la linea nera è la frontiera a probabilità 0,5.

Per il TFM ogni riquadro è una chiamata a `fit` (memorizza i punti) e una a `predict_proba` (su tutti i punti della griglia).

In [ ]:
def griglia_frontiere(X, y, dimensioni=DIMENSIONI, risoluzione=RISOLUZIONE):
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, risoluzione),
        np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, risoluzione),
    )
    punti = np.c_[xx.ravel(), yy.ravel()]
    nomi = list(modelli())
    fig, assi = plt.subplots(
        len(dimensioni), len(nomi), figsize=(3 * len(nomi), 3 * len(dimensioni)), squeeze=False)
    inizio = time.perf_counter()
    for i, n in enumerate(dimensioni):
        X_ctx, y_ctx = X[:n], y[:n]
        for j, (nome, modello) in enumerate(modelli().items()):
            proba = modello.fit(X_ctx, y_ctx).predict_proba(punti)[:, 1].reshape(xx.shape)
            ax = assi[i, j]
            ax.contourf(xx, yy, proba, levels=np.linspace(0, 1, 11), cmap="RdBu", alpha=0.8)
            ax.contour(xx, yy, proba, levels=[0.5], colors="k", linewidths=1)
            ax.scatter(X_ctx[:, 0], X_ctx[:, 1], c=y_ctx, cmap="RdBu", edgecolors="k", s=12)
            ax.set_xticks([])
            ax.set_yticks([])
            if i == 0:
                ax.set_title(nome)
            if j == 0:
                ax.set_ylabel(f"{n} punti di contesto")
    plt.tight_layout()
    plt.show()
    print(f"Tempo totale: {time.perf_counter() - inizio:.0f} s")


griglia_frontiere(X_moons, y_moons)

In [ ]:
griglia_frontiere(X_circles, y_circles)

## Da osservare

**Grafico da commentare:** la colonna del TFM, dall'alto verso il basso.

- Con 20 punti, quale modello propone la frontiera più plausibile? E quale la più frastagliata?
- La regressione logistica non migliora con più dati su `circles`: perché?
- Lontano dai dati, dove i colori sfumano verso il bianco, il modello è incerto. Quali modelli mostrano questa incertezza e quali no?
- Il TFM non ha mai visto lune né cerchi in pre-training: da dove viene la forma della sua frontiera?

## Esercizio

Aggiungere il 20% di etichette sbagliate e osservare se la frontiera si deforma.

Suggerimenti:

- scegliere a caso il 20% dei punti con `np.random.default_rng(0)` e invertirne l'etichetta (`1 - y`);
- ridisegnare la griglia con le nuove etichette;
- confrontare la colonna del TFM con quella del kNN.

In [ ]:
# TODO: costruire y_rumorosa invertendo il 20% delle etichette di y_moons

# TODO: griglia_frontiere(X_moons, y_rumorosa)